# Automated Codebook

The second attempt attempts to remove the recency bias seen in notebook books. Here we code each interview individually and then combine into one.

## 1. Imports

In [1]:
import os
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 2. Configuration

In [2]:
MODEL_NAME = "gpt-oss:120b"
TEMPERATURE = 0.3 # taken from Nyaaba et al (2026)
NUM_CTX = 32768  # Context used for prediction. Could push higher

NUM_INTERVIEWS = 6

# Chunking Settings
CHUNK_SIZE = 4000  # Roughly 800-1000 words per chunk to prevent "Lost in the Middle"
CHUNK_OVERLAP = 400  # Overlap across chunks

## 3. Prompt Templates

### 3.1. System prompt: A variation on Tree of Thought (ToT)

Five experts that analyse each chunk slightly differently with expert 5 coordinating.  Differs from standard ToT as no experts leave the room.

In [3]:
EXPERTS_SETUP = """
Five different experts are collaboratively identifying themes across interviews regarding the reuse of a
simulation model of a stroke pathway.

- Expert 1 focuses on barriers and challenges in reuse.
- Expert 2 focuses on what went well in using the model (facilitators, enablers, successes).
- Expert 3 focuses on what needs to be improved to enable reuse
- Expert 4 focuses on anything unexpected, surprising, contradictory, or outlier observations.
- Expert 5 acts as the Lead Synthesizer and Editor.

Expert 5 has a strict responsibility to:
1. aggressively merge overlapping or near-duplicate codes;
2. prevent codebook bloat;
3. preserve traceability across ALL interviews i.e. do not tracibility to early interviews;
4. keep only the most representative supporting evidence for each code;
5. ensure that codes remain distinct, meaningful, and reusable across interviews.
"""

### 3.2 Codebook prompt

The prompt will vary depending on the interview number. For interview 2 onwards the current codebook is appended. Implemented using `jinja2` if else. 

In [4]:
codebook_template = """
{{ experts_setup }}

{% if current_codebook %}
Task: 
Update the existing codebook below using the new transcript chunk. 

CURRENT CODEBOOK:
{{ current_codebook }}

Instructions:
1. Apply Existing: If new evidence fits an existing code, add the quote. 
2. Create New: Only create a new code if the idea cannot be absorbed into an existing one.
3. Consolidate (Expert 5): Aggressively merge overlapping codes. Delete weak or highly specific micro-codes. Keep the codebook compact.

Evidence Rules:
- Keep a MAXIMUM of 3 representative quotes per code. Replace weaker quotes with better ones.
- Ensure diversity of quotes across interviewees.
- Quotes must be exact and include the ID in brackets: [{{ interview_id }}] "exact quote"

Output:
Return ONLY the updated Markdown table with columns:  Expert Role | Emerging Code | Definition | Relevant Interviews |Source Evidence.

{% else %}
Task: 
Extract emerging codes from the transcript chunk below to create an initial codebook. Do not create a code for each observation.

Instructions:
1. Experts 1-4 independently identify relevant codes based on their assigned perspectives.
2. Refine (Expert 5): Review all candidate codes. Merge overlaps, remove redundancies, and tighten definitions to ensure codes are distinct and broadly applicable.

Evidence Rules:
- Keep a MAXIMUM of 3 representative quotes per code.
- Quotes must be exact and include the ID in brackets: [{{ interview_id }}] "exact quote"

Output:
Return ONLY a Markdown table with columns: Expert Role | Emerging Code | Definition | Relevant Interviews |Source Evidence.

{% endif %}

Transcript Chunk ({{ interview_id }}):
{{ transcript_chunk }}
"""

### 3.3. Combination prompt

In [5]:
synthesis_template = """
{{ experts_setup }}

Task: 
You are provided with several individual codebooks generated from separate interviews. 
Your goal is to synthesize these into a single, master codebook.

INDIVIDUAL CODEBOOKS:
{{ all_codebooks }}

Instructions:
1. Merge (Expert 5): Consolidate overlapping codes across the different interviews into single, robust codes.
2. Exhaustive Traceability: The "Relevant Interviews" column MUST list EVERY single interview where this code appeared. Do not drop any interview IDs. 
3. Concise Evidence: Provide a MAXIMUM of 3 quotes per code in the "Source Evidence" column. 
4. Critical Rule: It is fully expected and required that a code might list 6 interviews in the "Relevant Interviews" column, but only show 3 quotes in the "Source Evidence" column. Pick the 3 most diverse and powerful quotes available.

Output:
Return ONLY the finalized master Markdown table with columns: Expert Role | Emerging Code | Definition | Relevant Interviews | Source Evidence.
"""


### 3.3 Setup prompt templates

In [6]:
individual_interview_prompt = PromptTemplate.from_template(
    codebook_template, 
    template_format="jinja2"
)

In [7]:
synthesis_prompt = PromptTemplate.from_template(
    synthesis_template, 
    template_format="jinja2"
)

## 4. Initialise Ollama and LLM

In [8]:
llm = ChatOllama(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    num_ctx=NUM_CTX
)

## 5. Setup chain and chunkifier

In [9]:
# Define the Chain (LCEL)

# chain = filled in prompt template -> chosen LLM -> cleaned up response
analysis_chain = individual_interview_prompt | llm | StrOutputParser()

In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " ", ""]
)

## 6. Run analysis function

In [11]:
def run_analysis():
    # Store individual codebooks for the final synthesis
    all_interview_codebooks = {}
    
    # STEP 1: Code each interview separately (skipping existing ones)
    for i in range(1, NUM_INTERVIEWS + 1):
        filename = f"../data/interview_{i}.txt"
        output_filename = f"../output/codebook_interview_{i}_final.md"
        current_interview_id = filename.replace(".txt", "").replace("_", " ").title()

        # [NEW] Check if we already processed this interview
        if os.path.exists(output_filename):
            print(f"⏩ Skipping LLM processing. Loading existing codebook for {current_interview_id}...")
            with open(output_filename, 'r', encoding='utf-8') as f:
                all_interview_codebooks[current_interview_id] = f.read()
            continue

        # If it doesn't exist, check if the raw text is available
        if not os.path.exists(filename):
            print(f"File {filename} not found. Skipping...")
            continue
            
        with open(filename, 'r', encoding='utf-8') as f:
            interview_text = f.read()
            
        chunks = text_splitter.split_text(interview_text)
        print(f"Divided {filename} into {len(chunks)} chunks.")
        
        # Reset the codebook for the new interview
        interview_codebook = ""
        
        for chunk_idx, chunk_text in enumerate(chunks, 1):
            print(f" -> Processing {current_interview_id} - Chunk {chunk_idx}/{len(chunks)}...")
            
            interview_codebook = analysis_chain.invoke({
                "experts_setup": EXPERTS_SETUP,
                "current_codebook": interview_codebook,
                "transcript_chunk": chunk_text,
                "interview_id": current_interview_id
            })
            
        # Save the completed codebook for this specific interview
        with open(output_filename, 'w', encoding='utf-8') as f:
            f.write(interview_codebook)
            
        # Store for the final synthesis step
        all_interview_codebooks[current_interview_id] = interview_codebook
        print(f"✅ Finished coding {filename}.")

    # STEP 2: Combine all individual codebooks
    print("\n -> Starting final codebook synthesis...")
    
    # Combine all individual codebooks into a single text block
    combined_codebooks_text = ""
    for interview_id, codebook in all_interview_codebooks.items():
        combined_codebooks_text += f"\n### {interview_id} Codebook\n{codebook}\n"
    
    final_synthesis_chain = synthesis_prompt | llm | StrOutputParser()
    
    final_codebook = final_synthesis_chain.invoke({
        "experts_setup": EXPERTS_SETUP,
        "all_codebooks": combined_codebooks_text
    })
    
    with open("../output/final_codebook.md", 'w', encoding='utf-8') as f:
        f.write(final_codebook)
        
    print(f"\n ✅ Individual-Combination Analysis complete")

In [12]:
run_analysis()

⏩ Skipping LLM processing. Loading existing codebook for ../Data/Interview 1...
⏩ Skipping LLM processing. Loading existing codebook for ../Data/Interview 2...
⏩ Skipping LLM processing. Loading existing codebook for ../Data/Interview 3...
⏩ Skipping LLM processing. Loading existing codebook for ../Data/Interview 4...
⏩ Skipping LLM processing. Loading existing codebook for ../Data/Interview 5...
⏩ Skipping LLM processing. Loading existing codebook for ../Data/Interview 6...

 -> Starting final codebook synthesis...

 ✅ Individual-Combination Analysis complete
